In [1]:
import Utils.ab_makeData as abn
import peppi_py as pp


import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.pyplot import plot, xlim, ylim
import numpy


##
import pandas as pd

__init__ utils


## Model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
class Net(nn.Module):
    def __init__(self, input_size, output_size, layerSize):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(input_size, layerSize)
        self.fc2 = nn.Linear(layerSize,layerSize)
        self.fc3 = nn.Linear(layerSize,layerSize)
        self.fc4 = nn.Linear(layerSize,layerSize)
        self.fc5 = nn.Linear(layerSize,layerSize)
        self.fc6 = nn.Linear(layerSize, output_size)

    def forward(self, x):
        x = F.sigmoid(self.fc1(x))
        x = F.sigmoid(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        x = F.sigmoid(self.fc4(x))
        x = F.sigmoid(self.fc5(x))
        x = self.fc6(x)
        return x
    


In [ ]:
Smash1 = Net(4,2,32)


## Data

In [6]:
## Model
DATA_FOLDER = r"C:\Users\soph\PVV\Projects\Smash\shared_raw"
gameName = r'\02_32_35 [RUDE] Marth + [INFP] Marth (BF).slp'
test_path = DATA_FOLDER + r"\Slippi_Public_Dataset_v3" + gameName



In [7]:
startDF = pd.DataFrame(abn.make_pre_large(pp.read_slippi(test_path)))

22506


In [8]:
P0 = startDF[startDF['Player']=='0'][['x_pos', 'y_pos', 'x_joy', 'y_joy', 'frameNumber']]
P1 = startDF[startDF['Player']=='1'][['x_pos', 'y_pos', 'x_joy', 'y_joy', 'frameNumber']]

In [9]:
def inputStart(dfGood, dfBad):
   return dfGood.merge(dfBad.add_suffix(".enemy"), left_on='frameNumber', right_on='frameNumber.enemy')

preFinal = pd.concat([inputStart(P0, P1), inputStart(P1, P0)])
preFinal = preFinal.astype('float32')

print(preFinal.dtypes)
((preFinal))

x_pos                float32
y_pos                float32
x_joy                float32
y_joy                float32
frameNumber          float32
x_pos.enemy          float32
y_pos.enemy          float32
x_joy.enemy          float32
y_joy.enemy          float32
frameNumber.enemy    float32
dtype: object


,x_pos,y_pos,x_joy,y_joy,frameNumber,x_pos.enemy,y_pos.enemy,x_joy.enemy,y_joy.enemy,frameNumber.enemy
0,-38.799999,35.200001,0.000,0.0,0.0,38.799999,35.200001,0.0,0.0,0.0
1,-38.799999,35.200001,0.000,0.0,1.0,38.799999,35.200001,0.0,0.0,1.0
2,-38.799999,35.200001,0.000,0.0,2.0,38.799999,35.200001,0.0,0.0,2.0
3,-38.799999,35.200001,0.000,0.0,3.0,38.799999,35.200001,0.0,0.0,3.0
4,-38.799999,35.200001,0.000,0.0,4.0,38.799999,35.200001,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...
11248,-38.611015,2.400100,0.000,0.0,11248.0,-212.350800,76.275345,0.0,0.0,11248.0
11249,-38.611015,4.715100,0.000,0.0,11249.0,-214.999481,76.633156,0.0,0.0,11249.0
11250,-38.611015,6.945100,0.000,0.0,11250.0,-217.611481,76.955536,0.0,0.0,11250.0
11251,-38.611015,9.090100,0.675,0.0,11251.0,-220.186798,77.242493,0.0,0.0,11251.0


In [10]:
inputColumns = ['x_pos', 'y_pos', 'x_pos.enemy', 'y_pos.enemy']
outputColumns = ['x_joy', 'y_joy']

In [11]:
inputTensor = torch.tensor(preFinal[inputColumns].values)
targetTensor = torch.tensor(preFinal[outputColumns].values)

In [12]:
targetTensor

tensor([[0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        ...,
        [0.0000, 0.0000],
        [0.6750, 0.0000],
        [0.9750, 0.0000]])

## Train

In [13]:
Smash1.parameters
for name, p in Smash1.named_parameters():
    print(name, p.shape)
print(Smash1.fc1.bias.grad)

fc1.weight torch.Size([32, 4])
fc1.bias torch.Size([32])
fc2.weight torch.Size([32, 32])
fc2.bias torch.Size([32])
fc3.weight torch.Size([32, 32])
fc3.bias torch.Size([32])
fc4.weight torch.Size([32, 32])
fc4.bias torch.Size([32])
fc5.weight torch.Size([32, 32])
fc5.bias torch.Size([32])
fc6.weight torch.Size([2, 32])
fc6.bias torch.Size([2])
None


In [ ]:
t1 = Smash1(inputTensor)
t2 = targetTensor
lossFunction = nn.MSELoss()

lossFunction(t1,t2)

tensor(0.3111, grad_fn=<MseLossBackward0>)

In [15]:
t1.size()

torch.Size([22506, 2])

In [ ]:
## manually dumb train

EPOCHS = 100000

loss_list = []
for index in range(EPOCHS):

    currentTensor = Smash1(inputTensor)
    lossFunction = nn.MSELoss()
    loss = lossFunction(currentTensor, targetTensor)
    loss_list.append(loss)
    loss.backward()
    for p in Smash1.parameters():
        p.data.add_(-0.001 * p.grad)
        p.grad.data.zero_()

                    
        if index % 10 == 0 :
            print(index)
            print(p.grad)

In [17]:
loss_list

[tensor(0.3111, grad_fn=<MseLossBackward0>),
 tensor(0.3108, grad_fn=<MseLossBackward0>),
 tensor(0.3106, grad_fn=<MseLossBackward0>),
 tensor(0.3103, grad_fn=<MseLossBackward0>),
 tensor(0.3100, grad_fn=<MseLossBackward0>),
 tensor(0.3098, grad_fn=<MseLossBackward0>),
 tensor(0.3095, grad_fn=<MseLossBackward0>),
 tensor(0.3093, grad_fn=<MseLossBackward0>),
 tensor(0.3090, grad_fn=<MseLossBackward0>),
 tensor(0.3088, grad_fn=<MseLossBackward0>)]

In [18]:
savePath = "Model2.pt"
torch.save(Smash1.state_dict(), savePath)


In [19]:
newGuy = torch.load("Model1.pt", weights_only=True)
Smash2 = Net(4,2,32)
Smash2.load_state_dict(newGuy)
Smash2.eval()



Net(
  (fc1): Linear(in_features=4, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=32, bias=True)
  (fc6): Linear(in_features=32, out_features=2, bias=True)
)

In [20]:
import numpy as np
test_input = torch.tensor([1,2,3,4]).to(dtype=torch.float32)
Smash2(test_input).detach().numpy()

array([-0.02702457, -0.13033545], dtype=float32)

In [21]:
torch.accelerator.is_available()

True